# Tutorial: CNN 02 Image Classification

Audience:
- Learners who want a compact end-to-end CNN training example on image data.

Prerequisites:
- Module 06 on feedforward networks and Module 07 on stable training.

Learning goals:
- Train a small CNN on a real image dataset.
- Compare its performance with the structure of the task.
- Visualize learned filters and intermediate feature maps.


## Outline

1. Load and normalize the digits dataset.
2. Train a small convolutional classifier in PyTorch.
3. Evaluate predictions with accuracy and a confusion matrix.
4. Visualize first-layer filters and feature maps.


In [ ]:
from __future__ import annotations

import random
from dataclasses import dataclass

import matplotlib.pyplot as plt
import numpy as np
import torch
from sklearn.datasets import load_digits
from sklearn.metrics import ConfusionMatrixDisplay, accuracy_score
from sklearn.model_selection import train_test_split
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

SEED = 21
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device


In [ ]:
digits = load_digits()
X = digits.images.astype(np.float32) / 16.0
X = X[:, None, :, :]
y = digits.target.astype(np.int64)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=SEED, stratify=y
)

train_ds = TensorDataset(torch.tensor(X_train), torch.tensor(y_train))
test_ds = TensorDataset(torch.tensor(X_test), torch.tensor(y_test))
train_loader = DataLoader(train_ds, batch_size=64, shuffle=True)
test_loader = DataLoader(test_ds, batch_size=256)

X_train.shape, X_test.shape


## Step 1 - Define a compact CNN

The model uses two convolution blocks followed by a small classifier head. On `8x8` images, this is enough to show the core ideas without long training time.


In [ ]:
class SmallCNN(nn.Module):
    def __init__(self, num_classes: int = 10) -> None:
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 8, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(8, 16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(16 * 2 * 2, 32),
            nn.ReLU(),
            nn.Linear(32, num_classes),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.features(x)
        return self.classifier(x)

model = SmallCNN().to(device)
sum(p.numel() for p in model.parameters())


In [ ]:
@dataclass
class History:
    train_loss: list[float]
    test_accuracy: list[float]


def evaluate(model: nn.Module, loader: DataLoader) -> tuple[np.ndarray, np.ndarray]:
    model.eval()
    preds = []
    targets = []
    with torch.no_grad():
        for xb, yb in loader:
            logits = model(xb.to(device))
            preds.append(logits.argmax(dim=1).cpu().numpy())
            targets.append(yb.numpy())
    return np.concatenate(preds), np.concatenate(targets)


def train_model(model: nn.Module, train_loader: DataLoader, test_loader: DataLoader, epochs: int = 12) -> History:
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    criterion = nn.CrossEntropyLoss()
    history = History(train_loss=[], test_accuracy=[])

    for epoch in range(epochs):
        model.train()
        total_loss = 0.0
        total_items = 0
        for xb, yb in train_loader:
            xb = xb.to(device)
            yb = yb.to(device)
            optimizer.zero_grad()
            logits = model(xb)
            loss = criterion(logits, yb)
            loss.backward()
            optimizer.step()
            total_loss += loss.item() * xb.size(0)
            total_items += xb.size(0)

        preds, targets = evaluate(model, test_loader)
        history.train_loss.append(total_loss / total_items)
        history.test_accuracy.append(accuracy_score(targets, preds))

    return history

history = train_model(model, train_loader, test_loader, epochs=12)
history.test_accuracy[-1]


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3))
axes[0].plot(history.train_loss, marker="o")
axes[0].set_title("train loss")
axes[0].set_xlabel("epoch")
axes[0].set_ylabel("cross-entropy")

axes[1].plot(history.test_accuracy, marker="o")
axes[1].set_title("test accuracy")
axes[1].set_xlabel("epoch")
axes[1].set_ylabel("accuracy")
axes[1].set_ylim(0.0, 1.0)

plt.tight_layout()


## Step 2 - Evaluate the classifier

The confusion matrix shows which digit pairs remain difficult. Even on a small dataset, the CNN usually learns useful edge and stroke detectors quickly.


In [ ]:
preds, targets = evaluate(model, test_loader)
fig, ax = plt.subplots(figsize=(6, 6))
ConfusionMatrixDisplay.from_predictions(targets, preds, ax=ax, cmap="Blues", colorbar=False)
ax.set_title(f"test accuracy = {accuracy_score(targets, preds):.3f}")
plt.tight_layout()


## Step 3 - Visualize learned filters

The first convolution layer has shape `(out_channels, in_channels, kernel_height, kernel_width)`. Because the input is grayscale, each filter is one small `3x3` pattern.


In [ ]:
filters = model.features[0].weight.detach().cpu().numpy()
fig, axes = plt.subplots(2, 4, figsize=(8, 4))
for ax, kernel in zip(axes.flat, filters):
    ax.imshow(kernel[0], cmap="coolwarm")
    ax.axis("off")
plt.suptitle("learned conv1 filters")
plt.tight_layout()


## Step 4 - Visualize feature maps

Feature maps show the activations produced when the learned filters scan over an image.


In [ ]:
example_index = 0
example_image = torch.tensor(X_test[example_index : example_index + 1]).to(device)
with torch.no_grad():
    conv1_maps = model.features[0](example_image).cpu().numpy()[0]

fig, axes = plt.subplots(3, 3, figsize=(8, 8))
axes[0, 0].imshow(X_test[example_index, 0], cmap="gray")
axes[0, 0].set_title(f"input digit={y_test[example_index]}")
axes[0, 0].axis("off")
for ax, fmap in zip(axes.flat[1:], conv1_maps):
    ax.imshow(fmap, cmap="viridis")
    ax.axis("off")
plt.suptitle("conv1 feature maps")
plt.tight_layout()


## Exercises

- Replace max pooling with average pooling and compare accuracy.
- Increase the number of channels and check whether the filter visualizations become more specialized.
- Train an MLP with a similar parameter count and compare its error pattern with the CNN.
